In [1]:
import numpy as np
from sklearn.datasets import load_iris
import torch
import torch.nn as nn

In [2]:
iris = load_iris()

In [3]:
features = iris.data.astype(np.float64)
labels = iris.target

In [4]:
rng = np.random.default_rng(0)

train_i = []
test_i = []

In [5]:
for class_id in [0, 1, 2]:
    current_class_id = np.where(labels == class_id)[0]
    
    rng.shuffle(current_class_id)
    
    train_i.extend(current_class_id[:35])
    
    test_i.extend(current_class_id[35:])

In [6]:
x_train = features[train_i]
y_train = labels[train_i]

x_test = features[test_i]
y_test = labels[test_i]

In [7]:
train_mean = x_train.mean(axis=0)

train_std = x_train.std(axis=0, ddof=0)

x_train = (x_train - train_mean) / train_std

x_test = (x_test - train_mean) / train_std

In [8]:
rng_weights = np.random.default_rng(0)

w1_std = np.sqrt(2 / 4)
w1 = rng_weights.normal(loc=0.0, scale=w1_std, size=(4, 8)).astype(np.float64)

b1 = np.zeros(8, dtype=np.float64)

w2_std = np.sqrt(2 / (8 + 3))
w2 = rng_weights.normal(loc=0.0, scale=w2_std, size=(8, 3)).astype(np.float64)

b2 = np.zeros(3, dtype=np.float64)

In [9]:
def forward_backward(X, y, W1, b1, W2, b2, introduce_bug=True):
    N = X.shape[0] 

    Z1 = np.dot(X, W1) + b1
    
    A1 = np.maximum(0, Z1)

    Z2 = np.dot(A1, W2) + b2

    Z2_max = np.max(Z2, axis=1, keepdims=True)
    Z2_shifted = Z2 - Z2_max

    log_sum_exp = np.log(np.sum(np.exp(Z2_shifted), axis=1, keepdims=True))
    
    log_softmax = Z2_shifted - log_sum_exp

    probs = np.exp(log_softmax)

    correct_log_probs = log_softmax[np.arange(N), y]
    
    loss = -np.mean(correct_log_probs)

    dZ2 = probs.copy()
    dZ2[np.arange(N), y] -= 1
    if not introduce_bug:
            dZ2 /= N

    dW2 = np.dot(A1.T, dZ2)
    db2 = np.sum(dZ2, axis=0)

    dA1 = np.dot(dZ2, W2.T)

    dZ1 = dA1.copy()
    dZ1[Z1 <= 0] = 0

    dW1 = np.dot(X.T, dZ1)
    db1 = np.sum(dZ1, axis=0)

    gradients = {
        'W1': dW1,
        'b1': db1,
        'W2': dW2,
        'b2': db2
    }
    
    return loss, gradients

In [10]:
loss_np, grads_np = forward_backward(x_train, y_train, w1, b1, w2, b2)

In [11]:
X_pt = torch.tensor(x_train, dtype=torch.float64)
y_pt = torch.tensor(y_train, dtype=torch.long)

model = nn.Sequential(
    nn.Linear(4, 8),
    nn.ReLU(),
    nn.Linear(8, 3)
)
model = model.double()

with torch.no_grad():
    model[0].weight.copy_(torch.tensor(w1.T, dtype=torch.float64))
    model[0].bias.copy_(torch.tensor(b1, dtype=torch.float64))
    
    model[2].weight.copy_(torch.tensor(w2.T, dtype=torch.float64))
    model[2].bias.copy_(torch.tensor(b2, dtype=torch.float64))

criterion = nn.CrossEntropyLoss()

logits_pt = model(X_pt)
loss_pt = criterion(logits_pt, y_pt)

loss_pt.backward()

dW1_pt = model[0].weight.grad.numpy().T
db1_pt = model[0].bias.grad.numpy()
dW2_pt = model[2].weight.grad.numpy().T
db2_pt = model[2].bias.grad.numpy()

In [12]:
def check_diff(name, val_np, val_pt):

    max_diff = np.max(np.abs(val_np - val_pt))

    passed = "Так" if max_diff <= 1e-12 else "Ні"
    return f"{name:<15} | {max_diff:<45.3e} | {passed}"

print("Окремі значення втрати:")
print(f"NumPy Loss:   {loss_np:.15f}")
print(f"PyTorch Loss: {loss_pt.item():.15f}")
print("-" * 80)
print(f"{'Величина':<15} | {'Максимальна абсолютна різниця NumPy / PyTorch':<45} | Перевірку пройдено")
print("-" * 80)
print(check_diff("Втрата", loss_np, loss_pt.item()))
print(check_diff("Градієнт W1", grads_np['W1'], dW1_pt))
print(check_diff("Градієнт b1", grads_np['b1'], db1_pt))
print(check_diff("Градієнт W2", grads_np['W2'], dW2_pt))
print(check_diff("Градієнт b2", grads_np['b2'], db2_pt))

Окремі значення втрати:
NumPy Loss:   1.456200780114282
PyTorch Loss: 1.456200780114282
--------------------------------------------------------------------------------
Величина        | Максимальна абсолютна різниця NumPy / PyTorch | Перевірку пройдено
--------------------------------------------------------------------------------
Втрата          | 2.220e-16                                     | Так
Градієнт W1     | 1.852e+01                                     | Ні
Градієнт b1     | 1.564e+01                                     | Ні
Градієнт W2     | 3.419e+01                                     | Ні
Градієнт b2     | 1.120e+01                                     | Ні


In [13]:
epsilon = 1e-6

params_to_test = [
    ("W1[0,0]", w1, (0, 0), grads_np['W1'][0, 0]),
    ("b1[0]", b1, (0,), grads_np['b1'][0]),
    ("W2[0,0]", w2, (0, 0), grads_np['W2'][0, 0]),
    ("b2[0]", b2, (0,), grads_np['b2'][0])
]

print(f"{'Параметр':<10} | {'Градієнт backward()':<20} | {'Чисельна похідна':<20} | {'Абсолютна різниця':<20} | Перевірку пройдено")
print("-" * 105)

for name, param_array, idx, analytical_grad in params_to_test:
    orig_val = param_array[idx]
 
    param_array[idx] = orig_val + epsilon
    loss_plus, _ = forward_backward(x_train, y_train, w1, b1, w2, b2)
    
    param_array[idx] = orig_val - epsilon
    loss_minus, _ = forward_backward(x_train, y_train, w1, b1, w2, b2)
    
    param_array[idx] = orig_val
    
    numerical_grad = (loss_plus - loss_minus) / (2 * epsilon)
    
    diff = abs(analytical_grad - numerical_grad)
    
    passed = "Так" if diff <= 1e-8 else "Ні"
    
    print(f"{name:<10} | {analytical_grad:<20.10e} | {numerical_grad:<20.10e} | {diff:<20.3e} | {passed}")

Параметр   | Градієнт backward()  | Чисельна похідна     | Абсолютна різниця    | Перевірку пройдено
---------------------------------------------------------------------------------------------------------
W1[0,0]    | -2.0916193547e+00    | -1.9920184391e-02    | 2.072e+00            | Ні
b1[0]      | -7.4094417164e+00    | -7.0566111621e-02    | 7.339e+00            | Ні
W2[0,0]    | 1.4199171904e+01     | 1.3523020870e-01     | 1.406e+01            | Ні
b2[0]      | 8.9797727084e+00     | 8.5521644988e-02     | 8.894e+00            | Ні
